In [3]:
!pip3 install face_recognition

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 136.2 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 147.1 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for dlib: filename=dlib-19.24.2-cp311-cp311-linux_x86_64.whl size=4207110 sha256=77a64372f59f7de0f73177f5161b97ef65e618637c4772019d5b96cf77aebfca
  Stored in directory: /scratch/5799931.1.cds-gpu/pip-ephem-wheel-cache-1ini9er8/wheels/61/05/62/44b0bf18a0f8f9a0d65337b11237ecf12926d0d6e3807500bb
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566165 sha256=0113cf2c2ca2602c2e1211756804dee5734a91a96a95ba7846119ac12735ee98
  Stored in directory: /scratch/5799931.1.cds-gpu/pip-ephem-whe

In [1]:
import face_recognition
from face_recognition import face_locations
import os
import pandas as pd
from tqdm import tqdm
# I used mat73_to_pickle.py from https://github.com/emanuele/convert_matlab73_hdf5
from mat73_to_pickle import *

In [2]:
# set data paths
data_dir = './'
img_dir = os.path.join(data_dir, 'lfw')
indices_path = os.path.join(data_dir, 'indices_train_test.mat')
attr_path = os.path.join(data_dir, 'lfw_att_73.mat')

In [3]:
# read data (might take several minutes)
f = h5py.File(attr_path, mode='r')
data = recursive_dict(f)
df_label = pd.DataFrame(data['label'].T, columns=data['AttrName'], index=data['name'])
# change "\\" to "/"
df_label.index = [name.replace('\\', '/') for name in df_label.index]
df_label

,Male,Asian,White,Black,Baby,Child,Youth,Middle Aged,Senior,Black Hair,...,Pale Skin,5 o Clock Shadow,Strong Nose-Mouth Lines,Wearing Lipstick,Flushed Face,High Cheekbones,Brown Eyes,Wearing Earrings,Wearing Necktie,Wearing Necklace
Aaron_Eckhart/Aaron_Eckhart_0001.jpg,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
Aaron_Guiel/Aaron_Guiel_0001.jpg,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
Aaron_Patterson/Aaron_Patterson_0001.jpg,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
Aaron_Peirsol/Aaron_Peirsol_0001.jpg,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
Aaron_Peirsol/Aaron_Peirsol_0002.jpg,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Zorica_Radovic/Zorica_Radovic_0001.jpg,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
Zulfiqar_Ahmed/Zulfiqar_Ahmed_0001.jpg,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
Zumrati_Juma/Zumrati_Juma_0001.jpg,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
Zurab_Tsereteli/Zurab_Tsereteli_0001.jpg,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0


In [4]:
# extract face features using face_recognition.face_encodings
# take about 10 minutes on my pc
feature_vecs = []
fnames = []
for fname in tqdm(df_label.index):
    img_path = os.path.join(img_dir, fname)
    # face detection
    X_img = face_recognition.load_image_file(img_path)
    X_faces_loc = face_locations(X_img)
    # if the number of faces detected in a image is not 1, ignore the image
    if len(X_faces_loc) != 1:
        continue
    # extract 128 dimensional face features
    faces_encoding = face_recognition.face_encodings(X_img, known_face_locations=X_faces_loc)[0]
    feature_vecs.append(faces_encoding)
    fnames.append(fname)
df_feat = pd.DataFrame(feature_vecs, index=fnames)
df_label = df_label[df_label.index.isin(df_feat.index)]
df_feat.sort_index(inplace=True)
df_label.sort_index(inplace=True)

df_feat.to_csv('feature.csv')
df_label.to_csv('label.csv')

100%|██████████| 13143/13143 [15:33<00:00, 14.08it/s]
/scratch/5799931.1.cds-gpu/ipykernel_1139798/2702913278.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_label.sort_index(inplace=True)
